In [2]:
import pandas as pd
import numpy as np
import re

input_file = "Combined_dataset.csv"
df = pd.read_csv(input_file)
df.head()

,product_id,title,product_description,rating,ratings_count,initial_price,discount,final_price,currency,images,...,amount_of_stars,what_customers_said,seller_name,sizes,videos,seller_information,variations,best_offer,more_offers,category
0,8376765,Lino Perros,Women Navy Blue Solid Backpack,3.8,15,3995,58.0,"""₹3,995.00""",INR,http://assets.myntassets.com/assets/images/837...,...,"{""1_star"":2,""2_stars"":0,""3_stars"":3,""4_stars"":...",NaN,NaN,"[{""size"":""Onesize""}]","[""rw-8376765_cae700""]",NaN,[{}],{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
1,9136281,Tommy Hilfiger,Unisex Navy Blue Striped Backpack,4.5,67,2899,35.0,"""₹2,899.00""",INR,http://assets.myntassets.com/assets/images/913...,...,"{""1_star"":3,""2_stars"":4,""3_stars"":2,""4_stars"":...",NaN,NaN,"[{""size"":""Onesize""}]","[""rw-9136281_cae700""]",NaN,"[{},{}]",{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
2,17633752,Lavie,Aries Women Pink Mini Backpack,4.4,226,2999,65.0,"""₹2,999.00""",INR,http://assets.myntassets.com/assets/images/176...,...,"{""1_star"":9,""2_stars"":5,""3_stars"":10,""4_stars""...",NaN,NaN,"[{""size"":""S""}]","[""https://videos.myntassets.com/assets/videos/...",NaN,"[{},{},{},{},{},{}]",{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
3,1376949,F Gear,Unisex Navy & Grey Printed Burner Backpack,4.4,1052,1675,52.0,"""₹1,675.00""",INR,http://assets.myntassets.com/assets/images/137...,...,"{""1_star"":33,""2_stars"":21,""3_stars"":69,""4_star...",NaN,NaN,"[{""size"":""Onesize""}]","[""https://videos.myntassets.com/assets/videos/...",NaN,NaN,{},"[{""offer_name"":""10% Instant Discount on Citi C...",backpacks
4,13939916,MYTRIDENT,Men Blue Solid Bath Robe,4.7,12,2899,17.0,"""₹2,899.00""",INR,http://assets.myntassets.com/assets/images/pro...,...,"{""1_star"":0,""2_stars"":0,""3_stars"":0,""4_stars"":...",NaN,NaN,"[{""size"":""M"",""value"":""45.0in"",""value_name"":""Fr...",NaN,NaN,"[{},{},{},{},{},{},{},{},{},{},{},{},{},{}]",{},"[{""offer_name"":""10% Instant Discount on IndusI...",bath-robe


In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (1000, 24)

Columns:
['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category']

Data types:
product_id                  int64
title                         str
product_description           str
rating                    float64
ratings_count               int64
initial_price               int64
discount                  float64
final_price                   str
currency                      str
images                        str
delivery_options              str
product_details               str
breadcrumbs                   str
product_specifications        str
amount_of_stars               str
what_customers_said           str
seller_name                   str
sizes   

In [5]:
def parse_price(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('"', '').replace(',', '')
    s = re.sub(r'[^\d.]', '', s)
    return pd.to_numeric(s, errors='coerce')

df_clean = df.copy()
df_clean["price"] = df_clean["final_price"].apply(parse_price)
df_clean["initial_price"] = pd.to_numeric(df_clean["initial_price"], errors="coerce")
df_clean[["final_price", "price"]].head()

,final_price,price
0,"""₹3,995.00""",3995.0
1,"""₹2,899.00""",2899.0
2,"""₹2,999.00""",2999.0
3,"""₹1,675.00""",1675.0
4,"""₹2,899.00""",2899.0


In [6]:
text_fill_cols = [
    "product_description", "what_customers_said", "variations", "videos",
    "seller_information", "seller_name", "best_offer", "more_offers",
    "product_details", "product_specifications", "breadcrumbs", "images",
    "delivery_options", "sizes"
]

for col in text_fill_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Not available")

df_clean["discount"] = df_clean["discount"].fillna(df_clean["discount"].median())

print(df_clean.isna().sum().sort_values(ascending=False).head(10))

product_id             0
title                  0
product_description    0
rating                 0
ratings_count          0
initial_price          0
discount               0
final_price            0
currency               0
images                 0
dtype: int64


In [7]:
high_rating = df_clean[df_clean["rating"] >= 4]
selected_columns = df_clean[["title", "category", "price", "rating"]]

print("High-rated rows:", high_rating.shape[0])
selected_columns.head()

High-rated rows: 615


,title,category,price,rating
0,Lino Perros,backpacks,3995.0,3.8
1,Tommy Hilfiger,backpacks,2899.0,4.5
2,Lavie,backpacks,2999.0,4.4
3,F Gear,backpacks,1675.0,4.4
4,MYTRIDENT,bath-robe,2899.0,4.7


In [8]:
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = df_clean.shape[0]

print("Rows before:", before)
print("Rows after:", after)

print("Duplicates removed:", before - after)

Rows before: 1000
Rows after: 1000
Duplicates removed: 0


In [9]:
rng = np.random.default_rng(42)
df_clean["quantity"] = rng.integers(1, 6, size=len(df_clean))
df_clean["total_amount"] = df_clean["price"] * df_clean["quantity"]

df_clean[["price", "quantity", "total_amount"]].head()

,price,quantity,total_amount
0,3995.0,1,3995.0
1,2899.0,4,11596.0
2,2999.0,4,11996.0
3,1675.0,3,5025.0
4,2899.0,3,8697.0


In [10]:
output_file = "combined_dataset_cleaned.csv"
df_clean.to_csv(output_file, index=False)
print(f"Saved cleaned file to: {output_file}")
print("Final shape:", df_clean.shape)

Saved cleaned file to: combined_dataset_cleaned.csv
Final shape: (1000, 27)


In [ ]:
df.tail()

In [ ]:
df.isnull().sum()

In [11]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0
